# NB17 — Quantum Brain Architecture and Path Encoding

**Quantum Brain Research Laboratory — Alejandro Reynoso**


## Purpose

This notebook converts a governed Obsidian-style knowledge graph into the computational
objects required by a hybrid Quantum Brain. It does **not** claim that a quantum processor
reads language. The LLM compiles a question into seeds, targets, edge constraints and an
epistemic objective; the search engine operates on the resulting path space.

The notebook:

1. builds a typed, provenance-bearing synthetic vault;
2. enumerates admissible multi-hop reasoning paths;
3. attaches relevance, reliability, novelty, contradiction and cost features;
4. compares exact, beam, greedy and stochastic classical search;
5. creates the path catalogue consumed by NB18–NB21.

All data are synthetic. The notebook runs without an API key and can be adapted to a real
Obsidian vault by replacing `build_graph()` with a Markdown/front-matter parser.


In [ ]:
from pathlib import Path
import importlib.util, subprocess, sys

IN_COLAB = Path("/content").exists()
LAB_ROOT = Path("/content/Quantum_Brain_Lab") if IN_COLAB else Path.cwd() / "Quantum_Brain_Lab"
LAB_ROOT.mkdir(parents=True, exist_ok=True)
DEPS = LAB_ROOT / "_deps"

required = {
    "networkx": "networkx",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}
missing = [pip_name for module, pip_name in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    DEPS.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--target", str(DEPS), *missing]
    )
    sys.path.insert(0, str(DEPS))

print(f"Laboratory root: {LAB_ROOT}")


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import random
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

SEED = 271828
RNG = random.Random(SEED)
np.random.seed(SEED)

def stable_hash(value: Any) -> str:
    raw = json.dumps(value, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def cosine_words(a: str, b: str) -> float:
    wa = Counter(x.lower().strip(".,:;!?()") for x in a.split())
    wb = Counter(x.lower().strip(".,:;!?()") for x in b.split())
    keys = set(wa) | set(wb)
    if not keys:
        return 0.0
    va = np.array([wa[k] for k in keys], dtype=float)
    vb = np.array([wb[k] for k in keys], dtype=float)
    den = np.linalg.norm(va) * np.linalg.norm(vb)
    return float(va @ vb / den) if den else 0.0

def build_graph() -> nx.MultiDiGraph:
    """Synthetic governed investment-committee vault."""
    g = nx.MultiDiGraph(graph_version="QB-G0", title="Quantum Brain governed vault")
    nodes = {
        "AlphaBank": ("company", "Diversified bank with floating-rate loans, stable deposits, and legacy systems.", 0.00, 0.92),
        "BetaPayments": ("company", "Cloud-native payments platform with rapid growth, thin margins, and rich transaction data.", 0.20, 0.88),
        "ZetaCloud": ("company", "Enterprise cloud provider with recurring revenue and cyber concentration risk.", 0.05, 0.90),
        "GammaRetail": ("company", "Consumer retailer exposed to imported inventory and discretionary demand.", -0.10, 0.87),
        "DeltaLogistics": ("company", "Regional logistics network with pricing power and fuel exposure.", 0.10, 0.86),
        "EtaInsure": ("company", "Insurer benefiting from reinvestment yields while claims rise with inflation.", 0.00, 0.89),
        "RateShock": ("factor", "Policy rates rise sharply, increasing discount rates and funding costs.", -0.70, 0.96),
        "Regulation": ("factor", "Capital, data, conduct, and model-risk requirements tighten.", -0.40, 0.96),
        "CyberShock": ("factor", "A severe cyber event disrupts critical services and customer trust.", -0.80, 0.98),
        "ConsumerSlowdown": ("factor", "Real disposable income and discretionary demand weaken.", -0.70, 0.95),
        "FXShock": ("factor", "The peso depreciates and imported-input costs rise.", -0.60, 0.94),
        "AcquireBeta": ("decision", "AlphaBank considers acquiring BetaPayments.", 0.00, 0.95),
        "ExpandCredit": ("decision", "AlphaBank considers expanding unsecured consumer credit.", 0.00, 0.95),
        "MigrateCloud": ("decision", "The group considers migrating critical workloads to ZetaCloud.", 0.00, 0.95),
        "HedgeFX": ("decision", "GammaRetail considers increasing its foreign-exchange hedge ratio.", 0.00, 0.95),
        "E01": ("evidence", "Payments data can reduce fraud losses and improve cross-selling at AlphaBank.", 0.85, 0.91),
        "E02": ("evidence", "BetaPayments valuation is highly sensitive to higher discount rates.", -0.90, 0.94),
        "E03": ("evidence", "Legacy-control integration creates a material execution risk.", -0.80, 0.93),
        "E04": ("evidence", "Prior acquisitions performed better when product autonomy was preserved.", 0.70, 0.82),
        "E05": ("evidence", "Tighter data regulation increases the fixed cost of payments integration.", -0.75, 0.92),
        "E06": ("evidence", "A bank-payments data estate improves real-time risk detection.", 0.80, 0.89),
        "E07": ("evidence", "Unsecured credit losses rise nonlinearly in consumer slowdowns.", -0.95, 0.97),
        "E08": ("evidence", "Floating-rate assets initially benefit from higher rates.", 0.60, 0.88),
        "E09": ("evidence", "Deposit repricing can later compress the margin benefit.", -0.55, 0.90),
        "E10": ("evidence", "Independent model validation is required before credit expansion.", -0.65, 0.98),
        "E11": ("evidence", "Cloud migration reduces unit costs and improves analytic flexibility.", 0.75, 0.89),
        "E12": ("evidence", "Single-provider concentration can turn a cyber shock into a systemic outage.", -0.95, 0.97),
        "E13": ("evidence", "Workload segmentation contained a prior service disruption.", 0.65, 0.91),
        "E14": ("evidence", "Multi-cloud resilience reduces concentration but raises coordination cost.", 0.25, 0.85),
        "E15": ("evidence", "Layered FX hedges stabilize gross margin.", 0.75, 0.92),
        "E16": ("evidence", "Over-hedging destroys value if currency weakness reverses.", -0.65, 0.88),
        "E17": ("evidence", "FX collateral calls can create a temporary liquidity shock.", -0.55, 0.90),
        "WeakSignalA": ("signal", "A small merchant cohort is moving from cards to account-to-account payments.", 0.45, 0.67),
        "WeakSignalB": ("signal", "New cyber-insurance exclusions may transfer more outage risk to cloud clients.", -0.50, 0.69),
        "OutcomeAutonomy": ("outcome", "Preserved product autonomy accelerated customer migration in a prior deal.", 0.65, 0.94),
        "OutcomeCredit": ("outcome", "A prior downturn generated losses above the linear stress model.", -0.90, 0.96),
        "OutcomeCloud": ("outcome", "Segmentation reduced recovery time during an earlier outage.", 0.70, 0.95),
        "OutcomeHedge": ("outcome", "The hedge protected margin but triggered a collateral call.", 0.10, 0.95),
    }
    for node_id, (kind, text, polarity, reliability) in nodes.items():
        g.add_node(node_id, kind=kind, text=text, polarity=polarity,
                   reliability=reliability, status="authoritative",
                   source=f"source_{1 + len(node_id) % 9:02d}")
    edges = [
        ("AcquireBeta","AlphaBank","concerns"),("AcquireBeta","BetaPayments","concerns"),
        ("ExpandCredit","AlphaBank","concerns"),("MigrateCloud","ZetaCloud","concerns"),
        ("HedgeFX","GammaRetail","concerns"),("AlphaBank","RateShock","exposed_to"),
        ("AlphaBank","Regulation","exposed_to"),("BetaPayments","RateShock","exposed_to"),
        ("BetaPayments","Regulation","exposed_to"),("BetaPayments","ZetaCloud","depends_on"),
        ("ZetaCloud","CyberShock","exposed_to"),("GammaRetail","FXShock","exposed_to"),
        ("GammaRetail","ConsumerSlowdown","exposed_to"),("GammaRetail","DeltaLogistics","depends_on"),
        ("EtaInsure","RateShock","exposed_to"),("EtaInsure","ConsumerSlowdown","exposed_to"),
        ("RateShock","ConsumerSlowdown","causes"),("CyberShock","Regulation","causes"),
        ("FXShock","ConsumerSlowdown","causes"),
        ("E01","AcquireBeta","supports"),("E02","AcquireBeta","contradicts"),
        ("E03","AcquireBeta","contradicts"),("E04","AcquireBeta","supports"),
        ("E05","AcquireBeta","contradicts"),("E06","AcquireBeta","supports"),
        ("WeakSignalA","BetaPayments","informs"),("WeakSignalA","AcquireBeta","supports"),
        ("E04","OutcomeAutonomy","resulted_in"),("OutcomeAutonomy","AcquireBeta","supports"),
        ("E07","ExpandCredit","contradicts"),("E08","ExpandCredit","supports"),
        ("E09","ExpandCredit","contradicts"),("E10","ExpandCredit","contradicts"),
        ("E07","OutcomeCredit","resulted_in"),("OutcomeCredit","ExpandCredit","contradicts"),
        ("E11","MigrateCloud","supports"),("E12","MigrateCloud","contradicts"),
        ("E13","MigrateCloud","supports"),("E14","MigrateCloud","supports"),
        ("WeakSignalB","CyberShock","informs"),("WeakSignalB","MigrateCloud","contradicts"),
        ("E13","OutcomeCloud","resulted_in"),("OutcomeCloud","MigrateCloud","supports"),
        ("E15","HedgeFX","supports"),("E16","HedgeFX","contradicts"),
        ("E17","HedgeFX","contradicts"),("E15","OutcomeHedge","resulted_in"),
        ("OutcomeHedge","HedgeFX","supports"),
        ("E01","E06","corroborates"),("E03","Regulation","informs"),
        ("E05","Regulation","informs"),("E12","CyberShock","informs"),
        ("E17","FXShock","informs"),("E09","RateShock","informs"),
    ]
    for i, (u, v, rel) in enumerate(edges):
        g.add_edge(u, v, rel=rel, weight=0.65 + 0.35 * ((i % 7) / 6))
    return g

QUERIES = [
    {"id":"Q1","text":"Should AlphaBank acquire BetaPayments under higher rates and tighter regulation?",
     "target":"AcquireBeta","seeds":["RateShock","Regulation"],"risk":"high","expected":"caution"},
    {"id":"Q2","text":"Should AlphaBank expand unsecured credit during a consumer slowdown?",
     "target":"ExpandCredit","seeds":["ConsumerSlowdown","RateShock"],"risk":"high","expected":"caution"},
    {"id":"Q3","text":"How should critical workloads migrate to ZetaCloud under cyber risk?",
     "target":"MigrateCloud","seeds":["CyberShock","Regulation"],"risk":"high","expected":"qualified"},
    {"id":"Q4","text":"How should GammaRetail hedge foreign-exchange depreciation risk?",
     "target":"HedgeFX","seeds":["FXShock","ConsumerSlowdown"],"risk":"medium","expected":"qualified"},
]

def simple_graph(g: nx.MultiDiGraph) -> nx.Graph:
    h = nx.Graph()
    h.add_nodes_from(g.nodes(data=True))
    for u, v, d in g.edges(data=True):
        if h.has_edge(u, v):
            h[u][v]["weight"] += float(d.get("weight", 1.0))
        else:
            h.add_edge(u, v, weight=float(d.get("weight", 1.0)), rels={d.get("rel","related")})
    return h

def enumerate_paths(g: nx.MultiDiGraph, query: dict, cutoff: int = 5, cap: int = 80) -> list[list[str]]:
    h = simple_graph(g)
    starts = list(dict.fromkeys(query["seeds"] + [query["target"]]))
    destinations = [n for n, d in h.nodes(data=True)
                    if d.get("kind") in {"evidence","signal","outcome"}]
    paths = []
    for s in starts:
        for t in destinations:
            if s == t:
                continue
            try:
                for p in nx.all_simple_paths(h, s, t, cutoff=cutoff):
                    if query["target"] in p or any(seed in p for seed in query["seeds"]):
                        paths.append(p)
                        if len(paths) >= cap:
                            break
            except nx.NetworkXNoPath:
                pass
            if len(paths) >= cap:
                break
        if len(paths) >= cap:
            break
    unique = []
    seen = set()
    for p in paths:
        key = tuple(p)
        if key not in seen:
            seen.add(key); unique.append(p)
    return unique

def path_features(g: nx.MultiDiGraph, query: dict, path: list[str]) -> dict:
    attrs = [g.nodes[n] for n in path]
    text = " ".join(a.get("text","") for a in attrs)
    evidence = [a for a in attrs if a.get("kind") in {"evidence","outcome","signal"}]
    polarity = float(np.mean([a.get("polarity",0.0) for a in evidence])) if evidence else 0.0
    reliability = float(np.mean([a.get("reliability",0.5) for a in evidence])) if evidence else 0.5
    relevance = cosine_words(query["text"], text)
    kinds = {a.get("kind") for a in attrs}
    novelty = float(sum(a.get("kind") == "signal" for a in attrs) / max(1, len(path)))
    causal = float(any(g.nodes[n].get("kind") == "factor" for n in path)
                   and any(g.nodes[n].get("kind") in {"decision","outcome"} for n in path))
    contradiction = float(polarity < -0.15)
    support = float(polarity > 0.15)
    bridge = float(len(kinds) / 5.0)
    cost = len(path)
    score = (1.7*relevance + 0.9*reliability + 0.45*novelty + 0.40*causal
             + 0.35*bridge + 0.25*contradiction + 0.20*support - 0.06*cost)
    return {
        "relevance": relevance, "reliability": reliability, "novelty": novelty,
        "causal": causal, "contradiction": contradiction, "support": support,
        "bridge": bridge, "cost": cost, "polarity": polarity, "score": score,
    }

def build_catalog(g: nx.MultiDiGraph, queries: list[dict] = QUERIES) -> pd.DataFrame:
    rows = []
    for q in queries:
        for j, path in enumerate(enumerate_paths(g, q)):
            f = path_features(g, q, path)
            rows.append({"query_id":q["id"],"path_id":f"{q['id']}-P{j:03d}",
                         "path":" -> ".join(path),"nodes":path,**f})
    return pd.DataFrame(rows)

def json_graph(g: nx.MultiDiGraph) -> dict:
    return nx.node_link_data(g, edges="edges")

def load_graph(path: Path) -> nx.MultiDiGraph:
    return nx.node_link_graph(json.loads(path.read_text()), edges="edges",
                              directed=True, multigraph=True)

def ensure_baseline() -> tuple[nx.MultiDiGraph, pd.DataFrame]:
    graph_path = LAB_ROOT / "QB_baseline_graph.json"
    catalog_path = LAB_ROOT / "NB17_path_catalog.csv"
    if graph_path.exists() and catalog_path.exists():
        return load_graph(graph_path), pd.read_csv(catalog_path)
    g = build_graph()
    catalog = build_catalog(g)
    graph_path.write_text(json.dumps(json_graph(g), indent=2))
    catalog.assign(nodes=catalog["nodes"].apply(json.dumps)).to_csv(catalog_path, index=False)
    return g, catalog

def normalized_entropy(p: np.ndarray) -> float:
    p = np.asarray(p, dtype=float)
    p = p[p > 1e-15]
    return float(-(p*np.log(p)).sum()/np.log(max(2,len(p))))

def jensen_shannon(p: np.ndarray, q: np.ndarray) -> float:
    p = np.asarray(p,dtype=float); q=np.asarray(q,dtype=float)
    p=p/p.sum(); q=q/q.sum(); m=0.5*(p+q)
    def kl(a,b):
        mask=a>1e-15
        return float(np.sum(a[mask]*np.log(a[mask]/np.maximum(b[mask],1e-15))))
    return math.sqrt(max(0.0,0.5*kl(p,m)+0.5*kl(q,m)))

def portfolio_metrics(df: pd.DataFrame) -> dict:
    if df.empty:
        return {"paths":0,"mean_score":0,"contradiction_share":0,"support_share":0,
                "novelty":0,"node_coverage":0,"polarity_balance":0}
    nodes = set()
    for value in df["path"]:
        nodes.update(str(value).split(" -> "))
    return {
        "paths":len(df), "mean_score":float(df["score"].mean()),
        "contradiction_share":float(df["contradiction"].mean()),
        "support_share":float(df["support"].mean()),
        "novelty":float(df["novelty"].mean()), "node_coverage":len(nodes),
        "polarity_balance":float(1-abs(df["polarity"].mean())),
    }


## 1. Build the governed knowledge substrate


In [ ]:
g = build_graph()
catalog = build_catalog(g)
assert len(g) >= 35 and len(catalog) >= 40
(LAB_ROOT / "QB_baseline_graph.json").write_text(json.dumps(json_graph(g), indent=2))
catalog.assign(nodes=catalog["nodes"].apply(json.dumps)).to_csv(
    LAB_ROOT / "NB17_path_catalog.csv", index=False)
pd.DataFrame(QUERIES).assign(
    seeds=lambda d: d["seeds"].apply(json.dumps)
).to_csv(LAB_ROOT / "QB_queries.csv", index=False)
print(f"Graph: {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")
print(f"Admissible path catalogue: {len(catalog)} paths across {catalog.query_id.nunique()} queries")
catalog.head(8)[["query_id","path_id","path","score","polarity","reliability"]]


## 2. Path encoding

For each path \(p\), the laboratory constructs a feature vector

\[
\phi(p)=
(r_p,e_p,n_p,c_p,d_p,b_p,\ell_p),
\]

containing semantic relevance, evidence reliability, novelty, causal content,
contradiction value, bridge value, and path length. The scalar score is transparent
and deliberately contestable: later notebooks treat its weights as a governed policy,
not as a law of nature.


In [ ]:
summary = catalog.groupby("query_id").agg(
    paths=("path_id","count"), mean_score=("score","mean"),
    max_score=("score","max"), contradictory=("contradiction","sum"),
    supporting=("support","sum"), weak_signal_share=("novelty","mean"),
    mean_length=("cost","mean")
).round(3)
display(summary)


## 3. Classical counterfactual search baselines


In [ ]:
def select_methods(frame: pd.DataFrame, k: int = 6) -> dict[str,pd.DataFrame]:
    exact = frame.nlargest(k, "score")
    greedy = frame.sort_values(["relevance","reliability"], ascending=False).head(k)
    beam_pool = frame.nlargest(min(len(frame),3*k),"relevance")
    beam = beam_pool.nlargest(k,"score")
    weights = np.exp(3*(frame["score"]-frame["score"].max()).to_numpy())
    weights = weights/weights.sum()
    idx = np.random.choice(len(frame), size=min(k,len(frame)), replace=False, p=weights)
    stochastic = frame.iloc[idx]
    return {"exact_top_k":exact,"greedy_relevance":greedy,
            "beam_search":beam,"stochastic":stochastic}

rows=[]
selections={}
for qid, frame in catalog.groupby("query_id"):
    selections[qid]={}
    for method, chosen in select_methods(frame).items():
        selections[qid][method]=chosen["path_id"].tolist()
        rows.append({"query_id":qid,"method":method,**portfolio_metrics(chosen)})
baseline_metrics=pd.DataFrame(rows)
baseline_metrics.to_csv(LAB_ROOT/"NB17_classical_baselines.csv",index=False)
display(baseline_metrics.round(3))


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4.6))
for method, frame in baseline_metrics.groupby("method"):
    axes[0].plot(frame["query_id"],frame["mean_score"],marker="o",label=method)
    axes[1].plot(frame["query_id"],frame["polarity_balance"],marker="o",label=method)
axes[0].set_title("Path quality by search method"); axes[0].set_ylabel("Mean path score")
axes[1].set_title("Balance of supporting and challenging evidence"); axes[1].set_ylabel("Polarity balance")
for ax in axes: ax.grid(alpha=.25); ax.set_xlabel("Query")
axes[1].legend(fontsize=8,bbox_to_anchor=(1.02,1),loc="upper left")
plt.tight_layout()
plt.savefig(LAB_ROOT/"NB17_classical_search.png",dpi=180,bbox_inches="tight")
plt.show()


## 4. Encoding manifest and reproducibility record


In [ ]:
manifest={
    "notebook":"NB17","created_utc":utc_now(),"seed":SEED,
    "graph_version":g.graph.get("graph_version"),
    "graph_hash":stable_hash(json_graph(g)),
    "query_hash":stable_hash(QUERIES),
    "nodes":g.number_of_nodes(),"edges":g.number_of_edges(),
    "candidate_paths":len(catalog),
    "encoding":"explicit_path_feature_vector_v1",
    "status":"validated_synthetic_experiment",
}
(LAB_ROOT/"NB17_manifest.json").write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))


## Interpretation

The path catalogue is the bridge between language and quantum computation. Natural
language has been compiled into a finite, auditable combinatorial space. NB18 will
evolve an amplitude distribution over a candidate graph; NB19 will treat path selection
as a constrained evidence-portfolio problem.
